# Transformer encoder–decoder nhỏ

TV3 tự cài các khối attention, feed-forward và positional encoding theo kiến trúc gốc.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.corpus import records

sample = next(records("train"))
print(sample)

In [ ]:
import torch

def decoder_mask(target_ids, pad_id=0):
    length = target_ids.size(1)
    causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
    return causal[None, None, :, :] & (target_ids != pad_id)[:, None, None, :]

mask = decoder_mask(torch.tensor([[2, 15, 0]]))
assert mask.shape == (1, 1, 3, 3)
assert mask[0, 0, 1].tolist() == [True, True, False]
assert not mask[..., 2].any()

## Các ô cần hoàn thiện

Tách Q/K/V thành nhiều head, chia logits cho căn bậc hai của d_k, áp mask trước softmax. Encoder dùng source padding mask. Decoder dùng causal mask và cross-attention với source padding mask. Ghép residual, dropout, layer norm và FFN. Lưu trọng số cross-attention theo layer/head để vẽ.

In [ ]:
config = dict(d_model=256, heads=4, encoder_layers=3, decoder_layers=3,
              d_ff=1024, dropout=0.1, seed=42)
assert config["d_model"] % config["heads"] == 0